# NSL-KDD Exploratory Data Analysis

Exploration only -- every transform used downstream lives in `src/ids_anomaly/`, not here.
This notebook exists to (a) sanity-check the raw data and (b) generate the class-balance and
feature-distribution figures embedded in `docs/results.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ids_anomaly.data.download import download_raw
from ids_anomaly.data.preprocess import load_datasets
from ids_anomaly.data.schema import ALL_COLUMNS

sns.set_theme(style="whitegrid")
ASSETS = ROOT / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

download_raw(ROOT / "data" / "raw")
raw_train = pd.read_csv(ROOT / "data" / "raw" / "KDDTrain+.txt", header=None, names=ALL_COLUMNS)
raw_test = pd.read_csv(ROOT / "data" / "raw" / "KDDTest+.txt", header=None, names=ALL_COLUMNS)
raw_train.shape, raw_test.shape

In [ ]:
train_ds, test_ds, preprocessor = load_datasets(ROOT / "data" / "raw")
print("Preprocessed feature dim:", train_ds.X.shape[1])
print("Train attack fraction:", train_ds.is_attack.mean())
print("Test attack fraction:", test_ds.is_attack.mean())

## Attack category balance

Highly imbalanced -- U2R is a handful of examples, DoS dominates. This is *why* the per-category detection-rate breakdown in `docs/results.md` matters more than any single aggregate accuracy number.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, ds, title in zip(axes, [train_ds, test_ds], ["Train", "Test"], strict=True):
    order = ["normal", "dos", "probe", "r2l", "u2r"]
    counts = pd.Series(ds.attack_category).value_counts().reindex(order).fillna(0)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, hue=counts.index, legend=False, palette="viridis")
    ax.set_title(f"{title} split -- attack category counts")
    ax.set_yscale("log")
    ax.set_ylabel("count (log scale)")
fig.tight_layout()
fig.savefig(ASSETS / "attack_category_balance.png", dpi=150)

## Why the log1p + scale preprocessing choice

Raw byte/count columns are extremely heavy-tailed; left unscaled they would dominate any Euclidean-distance-based method.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(raw_train["src_bytes"].clip(upper=raw_train["src_bytes"].quantile(0.99)), ax=axes[0], bins=50)
axes[0].set_title("src_bytes (raw, clipped at p99)")
sns.histplot(np.log1p(raw_train["src_bytes"]), ax=axes[1], bins=50, color="darkorange")
axes[1].set_title("log1p(src_bytes)")
fig.tight_layout()
fig.savefig(ASSETS / "src_bytes_log1p_comparison.png", dpi=150)

In [ ]:
protocol_by_attack = pd.crosstab(raw_train["protocol_type"], raw_train["label"].ne("normal").map({True: "attack", False: "normal"}))
protocol_by_attack